In [ ]:
!pip install -U bitsandbytes accelerate "datasets<4.0.0"

In [ ]:
from datasets import load_dataset, DatasetDict
# Load the highest-agreement configuration
ds = load_dataset("takala/financial_phrasebank", "sentences_allagree")

train_val_split = ds['train'].train_test_split(test_size=0.2, seed=42)
val_test_split = train_val_split['test'].train_test_split(test_size=0.5, seed=42)

final_datasets = DatasetDict({
    'train': train_val_split['train'],
    'val': val_test_split['train'],
    'test': val_test_split['test']
})

In [ ]:
!pip install transformers torch

In [ ]:
import os
from transformers import AutoTokenizer

model_id = "meta-llama/Llama-2-7b-hf"

# Lấy token từ biến môi trường hệ thống
hf_token = os.getenv("HF_TOKEN")

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

# 2. Cấu hình padding cho Llama
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

# 3. Test thử
text = "Financial PhraseBank is a great dataset for sentiment analysis."
tokens = tokenizer(text)
print(tokens)

In [ ]:
def tokenize_func(examples):
    return tokenizer(
        examples['sentence'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized_ds = final_datasets.map(tokenize_func, batched=True)
print(tokenized_ds)

# QLORA

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
login(token=os.getenv("HF_TOKEN"))

model_id = "meta-llama/Llama-2-7b-hf"
num_classes = 3  # Thay đổi theo số lượng class bài toán của bạn


# 1. Tokenizer setup
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. BitsAndBytes Config (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 3. Load Model với num_labels và pad_token_id
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=num_classes,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token  # Đã thêm `token=` ở đây
)

model.config.pad_token_id = tokenizer.pad_token_id

# 4. Chuẩn bị model 4-bit để huấn luyện LoRA
model = prepare_model_for_kbit_training(model)

# 5. PEFT / LoRA Config chuẩn cho Classification
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=["score"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
!pip install evaluate scikit-learn

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

# 1. Data Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. Hàm tính metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).astype(np.float32).mean().item()
    return {"accuracy": accuracy}

# 3. Cấu hình TrainingArguments (Đã sửa eval_strategy)
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",  # Đã cập nhật theo phiên bản transformers mới
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    logging_steps=10,
    report_to="none"
)

# 4. Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['val'],
    processing_class=tokenizer,  # Thay thế cho tokenizer=tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 5. Chạy huấn luyện
trainer.train()

---
# Inference

In [ ]:
import torch
model.eval()
id2label = {0: "negative", 1: "neutral", 2: "positive"}

def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        ouputs=model(**inputs)
        logits=ouputs.logits
    predicted_class_id = torch.argmax(logits, dim=-1).item()
    return id2label[predicted_class_id]

test_text = "The company reported a significant increase in quarterly profits."
print(f"Câu: '{test_text}'")
print(f"Dự đoán cảm xúc: {predict_sentiment(test_text)}")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 1. Tokenize tập test (nếu chưa tokenize)
tokenized_test = tokenized_ds["test"]

# 2. Dùng trainer để dự đoán trên toàn bộ tập test
predictions_output = trainer.predict(tokenized_test)

# 3. Lấy ra logits (kết quả thô) và quy đổi thành nhãn dự đoán bằng argmax
logits = predictions_output.predictions
pred_labels = np.argmax(logits, axis=-1)

# 4. Lấy nhãn thật (true labels) từ tập test
true_labels = tokenized_test["label"]

# 5. Tính toán các chỉ số Accuracy và F1-score (macro hoặc weighted)
acc = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average="weighted") # Hoặc "macro"

print(f"Kết quả trên tập Test:")
print(f"- Accuracy: {acc:.4f}")
print(f"- F1-Score: {f1:.4f}")

In [ ]:
output_dir = "./my_llama_2_7b_finetuned"

# 1. Lưu model và tokenizer
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
import shutil

# Nén thư mục ra file zip ngay tại /kaggle/working/
shutil.make_archive(
    '/kaggle/working/my_llama_2_7b_finetuned',
    'zip',
    '/kaggle/working/my_llama_2_7b_finetuned',
)
print('Đã zip xong!')

---
# Full Finetune

In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

model_id = "meta-llama/Llama-2-7b-hf"

# Load nguyên bản mô hình (thường dùng bfloat16 hoặc float16 để đỡ tàn phá VRAM nhất có thể)
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, 
    num_labels=3, 
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-5,          # Theo yêu cầu đề bài (1e-5)
    per_device_train_batch_size=8, # Batch size (tùy VRAM GPU, có thể giảm xuống 4 nếu thiếu VRAM)
    per_device_eval_batch_size=8,
    num_train_epochs=3,          # Số epoch (3-5 theo đề bài)
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    fp16=True,                   # Dùng hỗn hợp độ chính xác để tiết kiệm VRAM
    report_to="none"             # Tắt wandb nếu không dùng
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['val'],
    processing_class=tokenizer,  # Thay thế cho tokenizer=tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


trainer.train()

In [ ]:
# ==========================================
# CELL HUẤN LUYỆN BERT-BASE-UNCASED
# ==========================================
import torch
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

# 1. Load Model & Tokenizer cho BERT
bert_model_id = "bert-base-uncased" # Hoặc "bert-base-multilingual-cased" cho tiếng Việt/đa ngôn ngữ
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_id)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_model_id,
    num_labels=3
)

# 2. Data Collator riêng cho BERT
bert_data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

# 3. Cấu hình TrainingArguments cho BERT
bert_training_args = TrainingArguments(
    output_dir="./results_bert",
    learning_rate=2e-5,                   # LR chuẩn cho fine-tuning BERT (2e-5 đến 5e-5)
    per_device_train_batch_size=16,       # BERT nhẹ nên có thể tăng batch size lên 16 hoặc 32
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),       # Bật fp16 nếu dùng GPU NVIDIA
    logging_steps=10,
    report_to="none"
)

# 4. Khởi tạo Trainer
bert_trainer = Trainer(
    model=bert_model,
    args=bert_training_args,
    train_dataset=tokenized_ds['train'],  # Đảm bảo tokenized_ds đã được tokenize bằng bert_tokenizer
    eval_dataset=tokenized_ds['val'],
    processing_class=bert_tokenizer,
    data_collator=bert_data_collator,
    compute_metrics=compute_metrics,
)

# 5. Tiến hành Huấn luyện
bert_trainer.train()